In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# STEP 1 — VALIDATE FIBEN BENCHMARK RESULTS
# ============================================================

base_results = Path(
    "/path/to/schemalens/fiben/results"
)

paths = {
    "sf1": base_results / "fiben_mongo_sf1" / "benchmark_aggregate_results.csv",
    "sf10": base_results / "fiben_mongo_sf10" / "benchmark_aggregate_results.csv",
    "sf30": base_results / "fiben_mongo_sf30" / "benchmark_aggregate_results.csv",
}

validation_rows = []

for expected_scale, path in paths.items():
    print("\n" + "=" * 80)
    print(f"Checking {expected_scale}")
    print("Path:", path)

    if not path.exists():
        print("ERROR: file not found")
        validation_rows.append({
            "scale": expected_scale,
            "file_exists": False,
            "scale_ok": False,
            "all_success": False,
            "q5_returns_docs": False,
            "q6_returns_docs": False,
            "no_null_metrics": False,
        })
        continue

    df = pd.read_csv(path)

    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    # Basic scale check
    scale_values = sorted(df["scale_label"].dropna().unique().tolist()) if "scale_label" in df.columns else []
    scale_ok = scale_values == [expected_scale]

    print("scale_label values:", scale_values)

    # Success check
    if "n_runs" in df.columns and "n_success_runs" in df.columns:
        all_success = (df["n_runs"] == df["n_success_runs"]).all()
    else:
        all_success = False

    # Hot only for semantic validation
    hot = df[df["run_phase"] == "hot"].copy()

    # Q5 and Q6 document-return checks
    q5 = hot[hot["query_name"] == "Q5_ReportsAndMetricDataOfCompany"]
    q6 = hot[hot["query_name"] == "Q6_TechUSListedSecuritiesWithHighLastTradedValue"]

    q5_avg_docs = q5["avg_documents_returned"].mean() if not q5.empty else 0
    q6_avg_docs = q6["avg_documents_returned"].mean() if not q6.empty else 0

    q5_returns_docs = q5_avg_docs > 0
    q6_returns_docs = q6_avg_docs > 0

    # Null metric check
    metric_cols = [
        "avg_latency_ms",
        "median_latency_ms",
        "p95_latency_ms",
        "p99_latency_ms",
        "avg_documents_returned",
        "avg_documents_written",
    ]
    existing_metric_cols = [c for c in metric_cols if c in df.columns]
    no_null_metrics = df[existing_metric_cols].isna().sum().sum() == 0

    print("all_success:", all_success)
    print("Q5 avg docs:", q5_avg_docs)
    print("Q6 avg docs:", q6_avg_docs)
    print("q5_returns_docs:", q5_returns_docs)
    print("q6_returns_docs:", q6_returns_docs)
    print("no_null_metrics:", no_null_metrics)

    print("\nRows by query and phase:")
    print(
        df.groupby(["query_name", "run_phase"])
        .size()
        .reset_index(name="rows")
        .to_string(index=False)
    )

    validation_rows.append({
        "scale": expected_scale,
        "file_exists": True,
        "n_rows": len(df),
        "scale_values": scale_values,
        "scale_ok": scale_ok,
        "all_success": all_success,
        "q5_avg_docs_hot": q5_avg_docs,
        "q6_avg_docs_hot": q6_avg_docs,
        "q5_returns_docs": q5_returns_docs,
        "q6_returns_docs": q6_returns_docs,
        "no_null_metrics": no_null_metrics,
    })

validation_df = pd.DataFrame(validation_rows)

print("\n" + "=" * 80)
print("FINAL VALIDATION SUMMARY")
display(validation_df)

print("\nDecision:")
for _, row in validation_df.iterrows():
    ok = (
        row["file_exists"]
        and row["scale_ok"]
        and row["all_success"]
        and row["q5_returns_docs"]
        and row["q6_returns_docs"]
        and row["no_null_metrics"]
    )

    print(f"{row['scale']}: {'OK' if ok else 'PROBLEM'}")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# STEP 2 — FIBEN INDIVIDUAL SF ANALYSIS
# Same output structure as IMDb/LDBC corrected analysis
# ============================================================

# ------------------------------------------------------------
# CONFIG — adjust only this block if needed
# ------------------------------------------------------------

base_results = Path(
    "/path/to/schemalens/fiben/results"
)

scale_dirs = {
    "sf1": base_results / "fiben_mongo_sf1",
    "sf10": base_results / "fiben_mongo_sf10",
    "sf30": base_results / "fiben_mongo_sf30",
}

DATASET_NAME = "fiben"
N_FULL_CONFIG_SPACE = 10
NEAR_BEST_THRESHOLD = 0.05
METRIC_COL = "p95_latency_ms"
GROUP_COL = "final_benchmark_group"

# ------------------------------------------------------------
# FIBEN query classification
# ------------------------------------------------------------

def extract_official_id(query_name):
    return pd.Series(query_name).str.extract(r"^(Q\d+)").iloc[0, 0]


def official_sort_key(qid):
    if pd.isna(qid):
        return 999
    return int(str(qid).replace("Q", ""))


def query_group(qid):
    if qid == "Q1":
        return "local_lookup"
    if qid == "Q2":
        return "shallow_rooted_retrieval"
    if qid == "Q3":
        return "associative_retrieval"
    if qid == "Q4":
        return "deep_associative_traversal"
    if qid == "Q5":
        return "analytical_containment_like"
    if qid == "Q6":
        return "filtered_search"
    if qid in ["Q7", "Q8"]:
        return "aggregation"
    if qid == "Q9":
        return "associative_comparison"
    if qid == "Q10":
        return "insert_update"
    return "other"


def semantic_family(qid):
    if qid in ["Q1", "Q2", "Q6"]:
        return "association"
    if qid in ["Q3", "Q4", "Q7", "Q8", "Q9"]:
        return "associative"
    if qid == "Q5":
        return "analytical_containment_like"
    if qid == "Q10":
        return "update"
    return "other"


def sort_g_classes(values):
    def key(v):
        v = str(v)
        if v == "CONTROL":
            return -1
        if v.startswith("G"):
            try:
                return int(v[1:])
            except Exception:
                return 99
        return 100
    return sorted([str(v) for v in values if pd.notna(v)], key=key)


# ------------------------------------------------------------
# Optional catalog support
# ------------------------------------------------------------
# The ideal situation is to use a real experiment catalog.
# If no catalog is found, the code falls back to:
# final_benchmark_group != "control"
#
# This fallback is acceptable for producing preliminary files,
# but for the final paper we should later confirm it against the
# real experiment catalog.

def find_catalog(results_dir):
    candidates = [
        results_dir / "mongo_experiment_catalog.csv",
        results_dir / "experiment_catalog.csv",
        results_dir / "benchmark_execution_plan.csv",
        base_results / "mongo_experiment_catalog.csv",
        base_results / "experiment_catalog.csv",
        base_results / "benchmark_execution_plan.csv",
    ]

    for path in candidates:
        if path.exists():
            return path

    return None


def load_catalog(results_dir):
    catalog_path = find_catalog(results_dir)

    if catalog_path is None:
        return None, "fallback_group_not_control"

    catalog = pd.read_csv(catalog_path)

    # Keep only useful columns when present
    keep_cols = []
    for col in [
        "candidate_id",
        "query_name",
        "g_class",
        "design_pattern",
        "final_benchmark_group",
        "benchmark_group",
        "semantic_family",
        "activated_family",
        "is_activated",
        "activated",
    ]:
        if col in catalog.columns:
            keep_cols.append(col)

    if "candidate_id" not in keep_cols:
        return None, "fallback_group_not_control"

    catalog = catalog[keep_cols].drop_duplicates()

    return catalog, str(catalog_path)


def attach_activation_info(hot, catalog, activation_source):
    hot = hot.copy()

    # Default fallback
    hot["is_activated"] = hot[GROUP_COL] != "control"
    hot["activation_source"] = activation_source

    if catalog is None:
        return hot

    merge_keys = ["candidate_id"]
    if "query_name" in catalog.columns and "query_name" in hot.columns:
        merge_keys = ["candidate_id", "query_name"]

    catalog_small = catalog.drop_duplicates(subset=merge_keys)

    hot = hot.merge(
        catalog_small,
        on=merge_keys,
        how="left",
        suffixes=("", "_catalog")
    )

    # Prefer explicit activation column if available
    if "is_activated_catalog" in hot.columns:
        hot["is_activated"] = hot["is_activated_catalog"].fillna(hot["is_activated"])
        hot["activation_source"] = "catalog:is_activated"
    elif "activated_catalog" in hot.columns:
        hot["is_activated"] = hot["activated_catalog"].fillna(hot["is_activated"])
        hot["activation_source"] = "catalog:activated"
    elif "final_benchmark_group_catalog" in hot.columns:
        hot["is_activated"] = hot["final_benchmark_group_catalog"].fillna(hot[GROUP_COL]) != "control"
        hot["activation_source"] = "catalog:final_benchmark_group"
    elif "benchmark_group_catalog" in hot.columns:
        hot["is_activated"] = hot["benchmark_group_catalog"].fillna(hot[GROUP_COL]) != "control"
        hot["activation_source"] = "catalog:benchmark_group"

    return hot


# ------------------------------------------------------------
# Main analysis for one scale
# ------------------------------------------------------------

def analyze_one_scale(scale_label, results_dir):
    print("\n" + "=" * 90)
    print(f"Analyzing {scale_label}")
    print("Results dir:", results_dir)

    agg_path = results_dir / "benchmark_aggregate_results.csv"
    if not agg_path.exists():
        raise FileNotFoundError(f"Missing file: {agg_path}")

    agg = pd.read_csv(agg_path)

    # Basic enrichment
    agg["official_id"] = agg["query_name"].str.extract(r"^(Q\d+)")
    agg["official_sort"] = agg["official_id"].apply(official_sort_key)
    agg["query_group"] = agg["official_id"].apply(query_group)
    agg["semantic_family"] = agg["official_id"].apply(semantic_family)

    # Use hot phase only
    hot = agg[agg["run_phase"] == "hot"].copy()

    if hot.empty:
        raise ValueError(f"No hot rows found for {scale_label}")

    # Attach catalog activation info if available
    catalog, activation_source = load_catalog(results_dir)
    hot = attach_activation_info(hot, catalog, activation_source)

    print("Activation source:", hot["activation_source"].iloc[0])
    print("Hot rows:", len(hot))

    rows = []

    for query_name, grp in hot.groupby("query_name"):
        grp = grp.copy().sort_values(METRIC_COL)

        best_all = grp.iloc[0]
        best_p95 = float(best_all[METRIC_COL])

        # Near-best within 5%
        grp["relative_loss_to_best"] = (grp[METRIC_COL] - best_p95) / best_p95
        grp["near_best_5pct"] = grp["relative_loss_to_best"] <= NEAR_BEST_THRESHOLD

        activated = grp[grp["is_activated"] == True].copy()
        primary = grp[grp[GROUP_COL] == "primary"].copy()

        if activated.empty:
            raise ValueError(f"No activated configs found for query {query_name}")

        best_activated = activated.loc[activated[METRIC_COL].idxmin()]

        if len(primary) > 0:
            best_primary = primary.loc[primary[METRIC_COL].idxmin()]
            primary_regret = (
                float(best_primary[METRIC_COL]) - best_p95
            ) / best_p95
            best_primary_config = best_primary["g_class"]
            best_primary_p95 = float(best_primary[METRIC_COL])
        else:
            best_primary = None
            primary_regret = np.nan
            best_primary_config = None
            best_primary_p95 = np.nan

        n_tested = grp["candidate_id"].nunique()
        n_activated = activated["candidate_id"].nunique()
        dsr = 1 - (n_activated / N_FULL_CONFIG_SPACE)

        activated_ids = set(activated["candidate_id"].astype(str))
        best_all_id = str(best_all["candidate_id"])

        top1_preserved = best_all_id in activated_ids

        top3 = grp.nsmallest(min(3, len(grp)), METRIC_COL)
        top3_ids = set(top3["candidate_id"].astype(str))
        top3_preserved = len(top3_ids.intersection(activated_ids)) > 0

        near_best = grp[grp["near_best_5pct"] == True]
        near_best_ids = set(near_best["candidate_id"].astype(str))
        near_best_preserved = len(near_best_ids.intersection(activated_ids)) > 0

        activated_regret = (
            float(best_activated[METRIC_COL]) - best_p95
        ) / best_p95

        rows.append({
            # Identity
            "scale_label": scale_label,
            "official_id": best_all["official_id"],
            "official_sort": best_all["official_sort"],
            "query_name": query_name,
            "query_group": best_all["query_group"],
            "semantic_family": best_all["semantic_family"],

            # Search space
            "n_tested_configs": n_tested,
            "n_activated_configs": n_activated,
            "DSR": dsr,

            # Tested/activated configs
            "tested_configs": ",".join(sort_g_classes(grp["g_class"].unique())),
            "activated_configs": ",".join(sort_g_classes(activated["g_class"].unique())),

            # Best global
            "best_candidate_id": best_all["candidate_id"],
            "best_config": best_all["g_class"],
            "best_group": best_all[GROUP_COL],
            "best_design_pattern": best_all["design_pattern"],
            "best_p95_ms": best_p95,

            # Activated quality
            "best_activated_candidate_id": best_activated["candidate_id"],
            "best_activated_config": best_activated["g_class"],
            "best_activated_group": best_activated[GROUP_COL],
            "best_activated_design_pattern": best_activated["design_pattern"],
            "best_activated_p95_ms": float(best_activated[METRIC_COL]),
            "top1_preserved_by_activated": top1_preserved,
            "top3_preserved_by_activated": top3_preserved,
            "activated_regret": activated_regret,

            # Near-best
            "n_near_best": int(near_best["candidate_id"].nunique()),
            "near_best_configs": ",".join(sort_g_classes(near_best["g_class"].unique())),
            "near_best_ratio": float(len(near_best) / len(grp)),
            "near_best_preserved_by_activated": near_best_preserved,

            # Primary quality
            "best_primary_config": best_primary_config,
            "best_primary_p95_ms": best_primary_p95,
            "primary_regret": primary_regret,

            # Diagnostics
            "activation_source": best_all["activation_source"],
        })

    analysis_df = (
        pd.DataFrame(rows)
        .sort_values(["official_sort", "query_name"])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Save main analysis file
    # --------------------------------------------------------

    main_output = results_dir / "schemalens_reduction_analysis_hot.csv"
    analysis_df.to_csv(main_output, index=False)

    # --------------------------------------------------------
    # Summary file
    # --------------------------------------------------------

    summary_df = pd.DataFrame([{
        "scale_label": scale_label,
        "n_queries": analysis_df["query_name"].nunique(),
        "avg_DSR": analysis_df["DSR"].mean(),
        "top1_preservation": analysis_df["top1_preserved_by_activated"].mean(),
        "top3_preservation": analysis_df["top3_preserved_by_activated"].mean(),
        "near_best_preservation": analysis_df["near_best_preserved_by_activated"].mean(),
        "mean_activated_regret": analysis_df["activated_regret"].mean(),
        "mean_primary_regret": analysis_df["primary_regret"].dropna().mean(),
        "primary_winners": int((analysis_df["best_group"] == "primary").sum()),
        "secondary_winners": int((analysis_df["best_group"] == "secondary_affected").sum()),
        "control_winners": int((analysis_df["best_group"] == "control").sum()),
        "activation_source": analysis_df["activation_source"].iloc[0],
    }])

    summary_output = results_dir / f"{DATASET_NAME}_summary_hot_catalog_corrected.csv"
    summary_df.to_csv(summary_output, index=False)

    # --------------------------------------------------------
    # Best by query-scale file
    # --------------------------------------------------------

    best_output = results_dir / f"{DATASET_NAME}_best_by_query_scale_hot_catalog_corrected.csv"
    analysis_df.to_csv(best_output, index=False)

    # --------------------------------------------------------
    # Secondary winners
    # --------------------------------------------------------

    secondary_winners = analysis_df[analysis_df["best_group"] == "secondary_affected"].copy()
    secondary_output = results_dir / f"{DATASET_NAME}_secondary_winners_hot_catalog_corrected.csv"
    secondary_winners.to_csv(secondary_output, index=False)

    # --------------------------------------------------------
    # Control winners
    # --------------------------------------------------------

    control_winners = analysis_df[analysis_df["best_group"] == "control"].copy()
    control_output = results_dir / f"{DATASET_NAME}_control_winners_hot_catalog_corrected.csv"
    control_winners.to_csv(control_output, index=False)

    # --------------------------------------------------------
    # Initial representative cases
    # These are candidates; we will confirm them in Step 3.
    # --------------------------------------------------------

    representative_ids = ["Q2", "Q4", "Q5"]

    representative_cases = analysis_df[
        analysis_df["official_id"].isin(representative_ids)
    ].copy()

    representative_cases["table12_dataset"] = "FIBEN"

    representative_cases["table12_family"] = representative_cases["official_id"].map({
        "Q2": "Association",
        "Q4": "Associative / analytical traversal",
        "Q5": "Analytical / containment-like",
    })

    representative_cases["table12_representative_query"] = representative_cases["query_name"]
    representative_cases["table12_activated_configs"] = representative_cases["activated_configs"]
    representative_cases["table12_best_config"] = representative_cases["best_config"]
    representative_cases["table12_DSR"] = representative_cases["DSR"]
    representative_cases["table12_top1"] = representative_cases["top1_preserved_by_activated"]
    representative_cases["table12_near_best"] = representative_cases["near_best_preserved_by_activated"]
    representative_cases["table12_regret"] = representative_cases["activated_regret"]

    representative_cases["table12_interpretation_draft"] = representative_cases["official_id"].map({
        "Q2": "Correct association family activated; shallow corporation-centred access was reduced to a small competitive family.",
        "Q4": "Correct bridge-oriented family activated; the reduction preserved competitive alternatives for account/security traversal.",
        "Q5": "Correct analytical family activated; the benchmark selected the strongest trade-off among reference, embedded, and hybrid alternatives.",
    })

    representative_output = results_dir / f"{DATASET_NAME}_representative_cases_hot.csv"
    representative_cases.to_csv(representative_output, index=False)

    # --------------------------------------------------------
    # Print concise diagnostics
    # --------------------------------------------------------

    print("\nSaved files:")
    print(main_output)
    print(summary_output)
    print(best_output)
    print(secondary_output)
    print(control_output)
    print(representative_output)

    print("\nSummary:")
    display(summary_df)

    print("\nRepresentative candidates:")
    display(
        representative_cases[
            [
                "scale_label",
                "official_id",
                "table12_family",
                "query_name",
                "activated_configs",
                "best_config",
                "best_group",
                "best_p95_ms",
                "DSR",
                "top1_preserved_by_activated",
                "near_best_preserved_by_activated",
                "activated_regret",
                "primary_regret",
            ]
        ]
    )

    return analysis_df, summary_df, representative_cases


# ============================================================
# Run for all scales
# ============================================================

all_analysis = {}
all_summaries = {}
all_representatives = {}

for scale_label, results_dir in scale_dirs.items():
    analysis_df, summary_df, representative_cases = analyze_one_scale(
        scale_label=scale_label,
        results_dir=results_dir
    )

    all_analysis[scale_label] = analysis_df
    all_summaries[scale_label] = summary_df
    all_representatives[scale_label] = representative_cases

print("\n" + "=" * 90)
print("STEP 2 FINISHED")

combined_summary = pd.concat(all_summaries.values(), ignore_index=True)
display(combined_summary)